In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metrics_df = pd.read_csv("metric.csv")
traces_df = pd.read_csv("trace.csv")
logs_df = pd.read_csv("log.csv")
errs_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes
metrics_df['timestamp'] = pd.to_datetime(metrics_df['timestamp'], unit='s', utc=True)
traces_df['timestamp'] = pd.to_datetime(traces_df['timestamp'], unit='s', utc=True)
logs_df['timestamp'] = pd.to_datetime(logs_df['timestamp'], unit='s', utc=True)
errs_df['timestamp'] = pd.to_datetime(errs_df['timestamp'], unit='s', utc=True)

# Ensure numeric 'value' where applicable
metrics_df['value'] = pd.to_numeric(metrics_df['value'], errors='coerce')
traces_df['value'] = pd.to_numeric(traces_df['value'], errors='coerce')
logs_df['value'] = pd.to_numeric(logs_df['value'], errors='coerce')

# Aggregate helper: compute global stats per series (do NOT filter by time)
metrics_agg = (
    metrics_df.dropna(subset=['value'])
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(
        count=('value', 'count'),
        p50=('value', 'median'),
        p95=('value', lambda x: x.quantile(0.95)),
        min=('value', 'min'),
        max=('value', 'max'),
        mean=('value', 'mean')
    )
    .reset_index()
)
metrics_agg['series_type'] = 'metric'

traces_agg = (
    traces_df.dropna(subset=['value'])
    .groupby(['cmdb_id', 'trace_name'])
    .agg(
        count=('value', 'count'),
        p50=('value', 'median'),
        p95=('value', lambda x: x.quantile(0.95)),
        min=('value', 'min'),
        max=('value', 'max'),
        mean=('value', 'mean')
    )
    .reset_index()
)
# Normalize column name to kpi_name for combined table
traces_agg = traces_agg.rename(columns={'trace_name': 'kpi_name'})
traces_agg['series_type'] = 'trace'

logs_agg = (
    logs_df.dropna(subset=['value'])
    .groupby(['cmdb_id', 'log_name'])
    .agg(
        count=('value', 'count'),
        p50=('value', 'median'),
        p95=('value', lambda x: x.quantile(0.95)),
        min=('value', 'min'),
        max=('value', 'max'),
        mean=('value', 'mean')
    )
    .reset_index()
)
logs_agg = logs_agg.rename(columns={'log_name': 'kpi_name'})
logs_agg['series_type'] = 'log'

# Combine summaries
agg_summary_df = pd.concat([metrics_agg, traces_agg, logs_agg], ignore_index=True)
# Ensure column order as requested
agg_summary_df = agg_summary_df[['series_type', 'cmdb_id', 'kpi_name', 'count', 'p50', 'p95', 'min', 'max', 'mean']]

# Top 50 by count (compact view)
agg_summary_df_top50 = agg_summary_df.sort_values('count', ascending=False).head(50)

# Unique cmdb_id lists with counts per file (compact)
metrics_cmdb_counts = metrics_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count')
traces_cmdb_counts = traces_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count')
logs_cmdb_counts = logs_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count')

# Return compact results (variables displayed by IPython)
agg_summary_df_top50, metrics_cmdb_counts, traces_cmdb_counts, logs_cmdb_counts

```
Out[1]:
```
```python
# Build a concise plain-English summary string from existing kernel variables
metrics_top_list = ", ".join(f"{r.cmdb_id}:{r.count}" for r in metrics_cmdb_counts.head(10).itertuples())
traces_top_list = ", ".join(f"{r.cmdb_id}:{r.count}" for r in traces_cmdb_counts.head(10).itertuples())
logs_top_list = ", ".join(f"{r.cmdb_id}:{r.count}" for r in logs_cmdb_counts.head(10).itertuples())

summary = (
    "Summary of telemetry loads and aggregates:\n"
    "- Files loaded and timestamps parsed to UTC: metric.csv, trace.csv, log.csv, error_logs.csv.\n\n"
    "- Combined summary (agg_summary_df_top50):\n"
    "  * Top 50 series by count are shown (the table returned). These top rows are primarily metric series\n"
    "    with per-series sample counts of 25. Example top metric entries include adservice metrics (cpu, diskio,\n"
    "    latency-50, latency-90, mem) and entries for paymentservice and productcatalogservice.\n\n"
    "- Unique cmdb_id counts (top items shown as cmdb_id:count):\n"
    f"  * metrics.csv: {metrics_top_list}\n"
    f"  * trace.csv:   {traces_top_list}\n"
    f"  * log.csv:     {logs_top_list}\n\n"
    "Conclusion: Data is loaded and aggregated. Metric series mostly contain 25 samples per KPI-series,\n"
    "while trace and log files have different distribution of samples per cmdb_id (see trace.csv and log.csv counts above).\n"
    "If you need, I can display more rows from the combined summary or the full per-file cmdb_id lists."
)

# Display the summary string as the final compact output
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   series_type                cmdb_id    kpi_name  count           p50           p95           min           max          mean
0       metric              adservice         cpu     25  7.772155e-01  8.961295e-01  6.943837e-01  9.058447e-01  7.869866e-01
1       metric              adservice      diskio     25  6.436678e+04  1.184518e+05  4.750311e+04  1.550786e+05  7.272644e+04
2       metric              adservice  latency-50     25  3.002772e-03  3.014567e-03  2.995540e-03  3.015392e-03  3.003981e-03
3       metric              adservice  latency-90     25  4.611902e-03  4.627793e-03  4.599742e-03  4.633901e-03  4.612684e-03
4       metric              adservice         mem     25  1.073528e+08  1.081521e+08  1.067306e+08  1.082437e+08  1.073544e+08
..         ...                    ...         ...    ...           ...           ...           ...           ...           ...
45      metric         paymentservice         mem     25  3.977530e+07  3.996934e+07  3.942257e+07  3.997846e+07  3.975027e+07
46      metric         paymentservice      socket     25  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00
47      metric         paymentservice    workload     25  9.133333e-01  1.196257e+00  5.644500e-01  1.307817e+00  9.345135e-01
48      metric  productcatalogservice         cpu     25  3.395262e+00  1.386276e+01  3.197929e+00  1.875374e+01  4.715612e+00
49      metric  productcatalogservice  latency-50     25  1.732268e-03  2.890383e-03  1.652536e-03  4.483068e-03  1.928679e-03

[50 rows x 9 columns],                   cmdb_id  count
0               adservice    175
1             cartservice    175
2         checkoutservice    175
3            emailservice    175
4   recommendationservice    175
5          paymentservice    175
6         currencyservice    150
7                frontend    150
8         shippingservice    150
9   productcatalogservice    150
10                  redis    100
11      frontend-external     25,                  cmdb_id  count
0        checkoutservice    800
1        frontendservice    700
2                   root    700
3  recommendationservice    500
4  productcatalogservice    400
5        currencyservice    300
6           emailservice    200
7         paymentservice    200,                  cmdb_id  count
0              adservice     50
1            cartservice     50
2        checkoutservice     50
3        currencyservice     50
4           emailservice     50
5               frontend     50
6         paymentservice     50
7  recommendationservice     50
8        shippingservice     50
9                  redis     10)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing full-series DataFrames: metrics_df, traces_df, logs_df, agg_summary_df
# Define incident window (UTC, inclusive)
window_start = pd.to_datetime("2024-01-17 17:06:07", utc=True)
window_end = pd.to_datetime("2024-01-17 17:36:07", utc=True)

# --- 1) Compute global thresholds (P95 and P05) from the full series (do NOT use windowed data) ---
# Metrics thresholds
metrics_thresh = (
    metrics_df.dropna(subset=['value'])
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(p95_full=lambda x: x.quantile(0.95), p05_full=lambda x: x.quantile(0.05))
    .reset_index()
)
metrics_thresh['series_type'] = 'metric'

# Traces thresholds (normalize trace_name -> kpi_name)
traces_thresh = (
    traces_df.dropna(subset=['value'])
    .rename(columns={'trace_name': 'kpi_name'})
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(p95_full=lambda x: x.quantile(0.95), p05_full=lambda x: x.quantile(0.05))
    .reset_index()
)
traces_thresh['series_type'] = 'trace'

# Logs thresholds (normalize log_name -> kpi_name)
logs_thresh = (
    logs_df.dropna(subset=['value'])
    .rename(columns={'log_name': 'kpi_name'})
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(p95_full=lambda x: x.quantile(0.95), p05_full=lambda x: x.quantile(0.05))
    .reset_index()
)
logs_thresh['series_type'] = 'log'

# Combine thresholds into one table
thresholds_df = pd.concat([metrics_thresh, traces_thresh, logs_thresh], ignore_index=True)
# Ensure column order
thresholds_df = thresholds_df[['series_type', 'cmdb_id', 'kpi_name', 'p95_full', 'p05_full']]

# --- 2) Filter full DataFrames to the incident window (inclusive) and normalize KPI column name ---
metrics_win = metrics_df[(metrics_df['timestamp'] >= window_start) & (metrics_df['timestamp'] <= window_end)].copy()
traces_win = traces_df[(traces_df['timestamp'] >= window_start) & (traces_df['timestamp'] <= window_end)].copy()
logs_win = logs_df[(logs_df['timestamp'] >= window_start) & (logs_df['timestamp'] <= window_end)].copy()

# Normalize kpi_name column
traces_win = traces_win.rename(columns={'trace_name': 'kpi_name'})
logs_win = logs_win.rename(columns={'log_name': 'kpi_name'})

metrics_win = metrics_win[['timestamp', 'cmdb_id', 'kpi_name', 'value']]
traces_win = traces_win[['timestamp', 'cmdb_id', 'kpi_name', 'value']]
logs_win = logs_win[['timestamp', 'cmdb_id', 'kpi_name', 'value']]

# --- 3) Helper to compute windowed stats per series using global thresholds ---
def compute_window_stats(win_df, series_type_label):
    # Merge window rows with thresholds to bring p95/p05 per row
    df = win_df.merge(
        thresholds_df[thresholds_df['series_type'] == series_type_label],
        on=['cmdb_id', 'kpi_name'],
        how='right',  # keep all series that have thresholds, even if no window rows
        suffixes=('', '_th')
    )
    # After right join, rows from thresholds with no window rows will have NaN in timestamp/value
    # Identify rows that actually have window samples
    has_sample = df['value'].notna()
    # For rows with samples, compute flags
    df['above95'] = False
    df['below05'] = False
    mask_samples = has_sample
    df.loc[mask_samples, 'above95'] = df.loc[mask_samples, 'value'] >= df.loc[mask_samples, 'p95_full']
    df.loc[mask_samples, 'below05'] = df.loc[mask_samples, 'value'] <= df.loc[mask_samples, 'p05_full']
    # For earliest timestamps where condition met, create conditional timestamp columns
    df['ts_above'] = df['timestamp'].where(df['above95'])
    df['ts_below'] = df['timestamp'].where(df['below05'])
    # Now group by series keys (series_type, cmdb_id, kpi_name) to compute aggregates
    group_cols = ['series_type', 'cmdb_id', 'kpi_name']
    agg = df.groupby(group_cols).agg(
        points_in_window = ('value', lambda x: x.count()),
        # global thresholds (take first since constant per group)
        p95 = ('p95_full', 'first'),
        p05 = ('p05_full', 'first'),
        count_above_p95 = ('above95', 'sum'),
        earliest_above_p95 = ('ts_above', 'min'),
        max_value = ('value', 'max'),
        count_below_p05 = ('below05', 'sum'),
        earliest_below_p05 = ('ts_below', 'min'),
        min_value = ('value', 'min')
    ).reset_index()
    # Ensure timestamp columns are timezone-aware datetimes (they already are); groups without hits will have NaT
    # Convert integer counts to int type
    int_cols = ['points_in_window','count_above_p95','count_below_p05']
    for c in int_cols:
        agg[c] = agg[c].astype(int)
    # Keep columns in requested order
    agg = agg[['series_type','cmdb_id','kpi_name','points_in_window','p95','p05',
               'count_above_p95','earliest_above_p95','max_value',
               'count_below_p05','earliest_below_p05','min_value']]
    return agg

# Compute per-series window summaries for each series type
metrics_window_summary = compute_window_stats(metrics_win, 'metric')
traces_window_summary = compute_window_stats(traces_win, 'trace')
logs_window_summary = compute_window_stats(logs_win, 'log')

# Combine into final series_anom_summary
series_anom_summary = pd.concat([metrics_window_summary, traces_window_summary, logs_window_summary], ignore_index=True)

# Compute a severity metric to sort by: total anomaly count in window
series_anom_summary['total_anom_count'] = series_anom_summary['count_above_p95'] + series_anom_summary['count_below_p05']

# Return top 20 rows ordered by total anomalies descending (compact)
series_anom_summary_top20 = series_anom_summary.sort_values('total_anom_count', ascending=False).head(20)

# Compact aggregation of anomaly counts by cmdb_id (sum of both counts), top 10
cmdb_anom_counts = (
    series_anom_summary.groupby('cmdb_id')[['count_above_p95','count_below_p05']]
    .sum()
    .assign(total_anom=lambda df: df['count_above_p95'] + df['count_below_p05'])
    .sort_values('total_anom', ascending=False)
    .reset_index()
)
cmdb_anom_counts_top10 = cmdb_anom_counts.head(10)

# Display the two compact results (IPython will show them)
series_anom_summary_top20, cmdb_anom_counts_top10

```
Out[2]:
```
Summary (plain English):

- Time window examined: 2024-01-17 17:06:07 – 2024-01-17 17:36:07 UTC.
- Many series had full coverage in the window (25 samples) and showed large numbers of anomalies (samples >= global P95 and <= global P05). The earliest anomaly timestamps in the top series are at 2024-01-17 17:09:00+00:00.
- Top per-series anomaly findings (examples from the top 20):
  - metric socket and diskio KPIs (multiple services: checkoutservice, cartservice, adservice, frontend, paymentservice, emailservice, currencyservice, recommendationservice, shippingservice, redis) show 25 samples above P95 and 25 below P05 in the window (total_anom_count = 50 for those series).
  - Several trace error_rate series for checkoutservice show the same pattern (25 above P95 + 25 below P05).
- Aggregated anomalies by service (sum of above-P95 and below-P05 counts) — top 10:
  1. checkoutservice: 674 total anomalies
  2. root: 539
  3. recommendationservice: 506
  4. frontendservice: 444
  5. emailservice: 333
  6. currencyservice: 331
  7. paymentservice: 319
  8. productcatalogservice: 312
  9. cartservice: 174
 10. adservice: 128
- Conclusion and suggested next steps:
  - The most impacted component is checkoutservice (largest anomaly count), with widespread anomalies also across frontend/recommendation/email/currency/payment services.
  - Primary suspicious signals are socket and diskio metrics and trace error_rate series. Investigate socket errors, disk I/O behavior, and trace error rates in checkoutservice first (check logs, network/socket errors, and downstream dependencies).
  - If you want, I can (a) show the full per-series anomaly rows for the top services, (b) list exact KPI names per service, or (c extract relevant raw log/error lines for the checkoutservice time-range.

The original code execution output of IPython Kernel is also provided below for reference:

(    series_type                cmdb_id                               kpi_name  points_in_window   p95   p05  count_above_p95        earliest_above_p95  max_value  count_below_p05        earliest_below_p05  min_value  total_anom_count
15       metric        checkoutservice                                 diskio                25   0.0   0.0               25 2024-01-17 17:09:00+00:00        0.0               25 2024-01-17 17:09:00+00:00        0.0                50
8        metric            cartservice                                 diskio                25   0.0   0.0               25 2024-01-17 17:09:00+00:00        0.0               25 2024-01-17 17:09:00+00:00        0.0                50
5        metric              adservice                                 socket                25   4.0   4.0               25 2024-01-17 17:09:00+00:00        4.0               25 2024-01-17 17:09:00+00:00        4.0                50
12       metric            cartservice                                 socket                25   4.0   4.0               25 2024-01-17 17:09:00+00:00        4.0               25 2024-01-17 17:09:00+00:00        4.0                50
38       metric               frontend                                 socket                25  28.0  28.0               25 2024-01-17 17:09:00+00:00       28.0               25 2024-01-17 17:09:00+00:00       28.0                50
46       metric         paymentservice                                 socket                25   3.0   3.0               25 2024-01-17 17:09:00+00:00        3.0               25 2024-01-17 17:09:00+00:00        3.0                50
42       metric         paymentservice                                 diskio                25   0.0   0.0               25 2024-01-17 17:09:00+00:00        0.0               25 2024-01-17 17:09:00+00:00        0.0                50
28       metric           emailservice                                 diskio                25   0.0   0.0               25 2024-01-17 17:09:00+00:00        0.0               25 2024-01-17 17:09:00+00:00        0.0                50
32       metric           emailservice                                 socket                25   3.0   3.0               25 2024-01-17 17:09:00+00:00        3.0               25 2024-01-17 17:09:00+00:00        3.0                50
25       metric        currencyservice                                 socket                25   4.0   4.0               25 2024-01-17 17:09:00+00:00        4.0               25 2024-01-17 17:09:00+00:00        4.0                50
19       metric        checkoutservice                                 socket                25   9.0   9.0               25 2024-01-17 17:09:00+00:00        9.0               25 2024-01-17 17:09:00+00:00        9.0                50
85        trace        checkoutservice    trace.to_checkoutservice.error_rate                25   0.0   0.0               25 2024-01-17 17:09:00+00:00        0.0               25 2024-01-17 17:09:00+00:00        0.0                50
73        trace        checkoutservice  trace.from_checkoutservice.error_rate                25   0.0   0.0               25 2024-01-17 17:09:00+00:00        0.0               25 2024-01-17 17:09:00+00:00        0.0                50
81        trace        checkoutservice             trace.from_root.error_rate                25   0.0   0.0               25 2024-01-17 17:09:00+00:00        0.0               25 2024-01-17 17:09:00+00:00        0.0                50
77        trace        checkoutservice  trace.from_frontendservice.error_rate                25   0.0   0.0               25 2024-01-17 17:09:00+00:00        0.0               25 2024-01-17 17:09:00+00:00        0.0                50
69       metric        shippingservice                                 socket                25   2.0   2.0               25 2024-01-17 17:09:00+00:00        2.0               25 2024-01-17 17:09:00+00:00        2.0                50
64       metric                  redis                                 socket                25   4.0   4.0               25 2024-01-17 17:09:00+00:00        4.0               25 2024-01-17 17:09:00+00:00        4.0                50
59       metric  recommendationservice                                 socket                25   4.0   4.0               25 2024-01-17 17:09:00+00:00        4.0               25 2024-01-17 17:09:00+00:00        4.0                50
55       metric  recommendationservice                                 diskio                25   0.0   0.0               25 2024-01-17 17:09:00+00:00        0.0               25 2024-01-17 17:09:00+00:00        0.0                50
109       trace        currencyservice  trace.from_frontendservice.error_rate                25   0.0   0.0               25 2024-01-17 17:09:00+00:00        0.0               25 2024-01-17 17:09:00+00:00        0.0                50,                  cmdb_id  count_above_p95  count_below_p05  total_anom
0        checkoutservice              338              336         674
1                   root              322              217         539
2  recommendationservice              263              243         506
3        frontendservice              218              226         444
4           emailservice              170              163         333
5        currencyservice              176              155         331
6         paymentservice              170              149         319
7  productcatalogservice              157              155         312
8            cartservice               87               87         174
9              adservice               64               64         128)```
```

In [ ]:
# In[3]:

import re
import pandas as pd

# Use existing series_anom_summary from kernel
# 1) Filter trace series with anomalies
traces_anom = series_anom_summary[
    (series_anom_summary['series_type'] == 'trace') &
    (series_anom_summary['total_anom_count'] > 0)
].copy()

# Prepare top-20 trace series view with requested columns (rename kpi_name -> trace_name)
traces_top20 = (
    traces_anom.assign(trace_name=traces_anom['kpi_name'])
    .loc[:, ['cmdb_id', 'trace_name', 'points_in_window',
             'count_above_p95', 'earliest_above_p95',
             'count_below_p05', 'earliest_below_p05',
             'total_anom_count', 'max_value', 'min_value']]
    .sort_values('total_anom_count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 2) Extract service tokens from trace_name and aggregate by token
def extract_token(name):
    s = str(name)
    # handle compact 'fromroot' (no underscore)
    if 'fromroot' in s:
        return 'root'
    # generic 'to_<service>' or 'from_<service>'
    m = re.search(r'(?:to_|from_)([a-zA-Z0-9]+)', s)
    if m:
        return m.group(1)
    # fallback: 'from<service>' (no underscore)
    m2 = re.search(r'\bfrom([a-zA-Z0-9]+)\b', s)
    if m2:
        return m2.group(1)
    return None

traces_anom['token'] = traces_anom['kpi_name'].apply(extract_token)

# compute per-row earliest anomaly timestamp (min of earliest_above_p95 and earliest_below_p05)
traces_anom['earliest_anom_ts'] = traces_anom[['earliest_above_p95','earliest_below_p05']].min(axis=1)

# keep only rows with a parsed token
token_df = traces_anom.dropna(subset=['token']).copy()

# aggregate by token
token_agg = (
    token_df.groupby('token').agg(
        total_anom_count = ('total_anom_count', 'sum'),
        earliest_anomaly_timestamp = ('earliest_anom_ts', 'min')
    )
    .reset_index()
    .sort_values('total_anom_count', ascending=False)
)

token_agg_top10 = token_agg.head(10).reset_index(drop=True)

# Return compact results
traces_top20, token_agg_top10

```
Out[3]:
```
Summary — plain English

- Window: 2024-01-17 17:06:07 → 2024-01-17 17:36:07 UTC.

1) Top trace series (high-level):
- The top 20 trace series with anomalies are almost entirely error_rate traces (names like trace.from_checkoutservice.error_rate, trace.to_checkoutservice.error_rate, trace.from_root.error_rate, etc.).
- Each listed trace series had full coverage in the window (25 points) and showed 25 samples >= global P95 and 25 samples <= global P05 (total_anom_count = 50 per series).
- Earliest anomaly seen for these traces is 2024-01-17 17:09:00+00:00.
- Observed max/min values for these traces in the window are 0.0 (the trace error_rate values in the returned rows are 0.0).

2) Aggregated tokens (parsed from trace names) — top tokens by total anomalies:
- root: 539 anomalies — earliest anomaly 2024-01-17 17:09:00+00:00
- checkoutservice: 500 anomalies — earliest 2024-01-17 17:09:00+00:00
- frontendservice: 444 anomalies — earliest 2024-01-17 17:09:00+00:00
- recommendationservice: 332 anomalies — earliest 2024-01-17 17:09:00+00:00
- productcatalogservice: 278 anomalies — earliest 2024-01-17 17:09:00+00:00
- currencyservice: 207 anomalies — earliest 2024-01-17 17:09:00+00:00
- emailservice: 145 anomalies — earliest 2024-01-17 17:09:00+00:00
- paymentservice: 145 anomalies — earliest 2024-01-17 17:09:00+00:00

Key takeaway and next steps:
- Trace error_rate across many services spiked (or showed extreme behavior) starting at ~2024-01-17 17:09:00 UTC. The most impacted tokens/services are root, checkoutservice, frontendservice and recommendationservice.
- Immediate next actions: investigate checkoutservice and the "root" caller around 17:09:00 UTC (check detailed traces and application logs, look for network/socket errors or downstream failures), and then examine frontend → checkout → recommendation call paths. I can fetch the exact trace entries or related logs for those services/time range if you want.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                 trace_name  points_in_window  count_above_p95        earliest_above_p95  count_below_p05        earliest_below_p05  total_anom_count  max_value  min_value
0         checkoutservice      trace.from_checkoutservice.error_rate                25               25 2024-01-17 17:09:00+00:00               25 2024-01-17 17:09:00+00:00                50        0.0        0.0
1         checkoutservice      trace.from_frontendservice.error_rate                25               25 2024-01-17 17:09:00+00:00               25 2024-01-17 17:09:00+00:00                50        0.0        0.0
2         checkoutservice        trace.to_checkoutservice.error_rate                25               25 2024-01-17 17:09:00+00:00               25 2024-01-17 17:09:00+00:00                50        0.0        0.0
3         checkoutservice                 trace.from_root.error_rate                25               25 2024-01-17 17:09:00+00:00               25 2024-01-17 17:09:00+00:00                50        0.0        0.0
4         checkoutservice        trace.to_currencyservice.error_rate                25               25 2024-01-17 17:09:00+00:00               25 2024-01-17 17:09:00+00:00                50        0.0        0.0
5         checkoutservice           trace.to_emailservice.error_rate                25               25 2024-01-17 17:09:00+00:00               25 2024-01-17 17:09:00+00:00                50        0.0        0.0
6                    root        trace.to_checkoutservice.error_rate                25               25 2024-01-17 17:09:00+00:00               25 2024-01-17 17:09:00+00:00                50        0.0        0.0
7   recommendationservice  trace.to_recommendationservice.error_rate                25               25 2024-01-17 17:09:00+00:00               25 2024-01-17 17:09:00+00:00                50        0.0        0.0
8         frontendservice                 trace.from_root.error_rate                25               25 2024-01-17 17:09:00+00:00               25 2024-01-17 17:09:00+00:00                50        0.0        0.0
9         frontendservice        trace.to_checkoutservice.error_rate                25               25 2024-01-17 17:09:00+00:00               25 2024-01-17 17:09:00+00:00                50        0.0        0.0
10        frontendservice        trace.to_frontendservice.error_rate                25               25 2024-01-17 17:09:00+00:00               25 2024-01-17 17:09:00+00:00                50        0.0        0.0
11        frontendservice        trace.to_currencyservice.error_rate                25               25 2024-01-17 17:09:00+00:00               25 2024-01-17 17:09:00+00:00                50        0.0        0.0
12         paymentservice      trace.from_checkoutservice.error_rate                25               25 2024-01-17 17:09:00+00:00               25 2024-01-17 17:09:00+00:00                50        0.0        0.0
13         paymentservice                 trace.from_root.error_rate                25               25 2024-01-17 17:09:00+00:00               25 2024-01-17 17:09:00+00:00                50        0.0        0.0
14        frontendservice  trace.to_recommendationservice.error_rate                25               25 2024-01-17 17:09:00+00:00               25 2024-01-17 17:09:00+00:00                50        0.0        0.0
15        frontendservice  trace.to_productcatalogservice.error_rate                25               25 2024-01-17 17:09:00+00:00               25 2024-01-17 17:09:00+00:00                50        0.0        0.0
16        currencyservice                 trace.from_root.error_rate                25               25 2024-01-17 17:09:00+00:00               25 2024-01-17 17:09:00+00:00                50        0.0        0.0
17           emailservice      trace.from_checkoutservice.error_rate                25               25 2024-01-17 17:09:00+00:00               25 2024-01-17 17:09:00+00:00                50        0.0        0.0
18        frontendservice      trace.from_frontendservice.error_rate                25               25 2024-01-17 17:09:00+00:00               25 2024-01-17 17:09:00+00:00                50        0.0        0.0
19           emailservice                 trace.from_root.error_rate                25               25 2024-01-17 17:09:00+00:00               25 2024-01-17 17:09:00+00:00                50        0.0        0.0,                    token  total_anom_count earliest_anomaly_timestamp
0                   root               539  2024-01-17 17:09:00+00:00
1        checkoutservice               500  2024-01-17 17:09:00+00:00
2        frontendservice               444  2024-01-17 17:09:00+00:00
3  recommendationservice               332  2024-01-17 17:09:00+00:00
4  productcatalogservice               278  2024-01-17 17:09:00+00:00
5        currencyservice               207  2024-01-17 17:09:00+00:00
6           emailservice               145  2024-01-17 17:09:00+00:00
7         paymentservice               145  2024-01-17 17:09:00+00:00)```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse existing DataFrames: logs_df, errs_df
# Define incident window (UTC, inclusive)
window_start = pd.to_datetime("2024-01-17 17:06:07", utc=True)
window_end = pd.to_datetime("2024-01-17 17:36:07", utc=True)

# 1) logs_df: filter for checkoutservice in window and aggregate by log_name
logs_checkout = logs_df[
    (logs_df['cmdb_id'] == 'checkoutservice') &
    (logs_df['timestamp'] >= window_start) &
    (logs_df['timestamp'] <= window_end)
].copy()

# Ensure numeric values for aggregation
logs_checkout['value'] = pd.to_numeric(logs_checkout['value'], errors='coerce')

logs_checkout_agg = (
    logs_checkout.groupby('log_name')['value']
    .agg(
        points_in_window = 'count',
        mean_value_in_window = 'mean',
        min_value_in_window = 'min',
        max_value_in_window = 'max'
    )
    .reset_index()
    # keep compact ordering: sort by points desc then log_name
    .sort_values(['points_in_window','log_name'], ascending=[False, True])
)

# 2) error_logs.csv: filter errs_df for checkoutservice in window, select up to 50 rows sorted ascending
errs_checkout = errs_df[
    (errs_df['cmdb_id'] == 'checkoutservice') &
    (errs_df['timestamp'] >= window_start) &
    (errs_df['timestamp'] <= window_end)
].copy()

errs_checkout_sorted = errs_checkout.sort_values('timestamp', ascending=True).head(50).reset_index(drop=True)

# Keyword scanning (case-insensitive substring matches)
keywords = ['socket', 'connection', 'timeout', 'timed out', 'broken pipe', 'refused',
            'disk', 'io', 'i/o', 'error', 'exception']

# Build keyword summary rows
rows = []
for kw in keywords:
    # Use case-insensitive substring search; use regex=False for literal matching (handles 'i/o')
    mask = errs_checkout['message'].str.contains(kw, case=False, na=False, regex=False)
    count = int(mask.sum())
    if count > 0:
        earliest_ts = errs_checkout.loc[mask, 'timestamp'].min()
        # pick first example message (sorted by timestamp)
        example_msg = errs_checkout.loc[mask].sort_values('timestamp', ascending=True).iloc[0]['message']
        example_msg_trim = (example_msg[:200]) if pd.notna(example_msg) else ""
    else:
        earliest_ts = pd.NaT
        example_msg_trim = ""
    rows.append({
        'keyword': kw,
        'count': count,
        'earliest_timestamp': earliest_ts,
        'example_message': example_msg_trim
    })

errs_keyword_summary = pd.DataFrame(rows).sort_values('count', ascending=False).reset_index(drop=True)

# Return the compact outputs
logs_checkout_agg, errs_checkout_sorted[['timestamp','cmdb_id','message']], errs_keyword_summary

```
Out[4]:
```
Summary — plain English

- Window: 2024-01-17 17:06:07 → 2024-01-17 17:36:07 UTC for cmdb_id = "checkoutservice".

1) log.csv (checkoutservice, window)
- log.error_count: 25 samples, mean=0.00, min=0, max=0 (no error_count recorded across samples).
- log.row_count: 25 samples, mean=81.72, min=12, max=114.

2) error_logs.csv (checkoutservice, window)
- No error_log rows found for checkoutservice in the window (empty result).
- Keyword scan (case-insensitive) for ['socket','connection','timeout','timed out','broken pipe','refused','disk','io','i/o','error','exception'] returned zero matches (all counts = 0).

Interpretation / next steps
- There are no textual error log entries for checkoutservice during the incident window, and the log.error_count series is all zeros, even though traces/metrics showed anomalies earlier. Investigate further by:
  - Checking logs for upstream/downstream services (frontend, recommendation, root) and centralized logging ingestion.
  - Increasing log verbosity or ensuring checkoutservice log collection is complete.
  - Correlating trace IDs/timestamps from traces to identify failing spans and related services.

The original code execution output of IPython Kernel is also provided below for reference:

(          log_name  points_in_window  mean_value_in_window  min_value_in_window  max_value_in_window
0  log.error_count                25                  0.00                    0                    0
1    log.row_count                25                 81.72                   12                  114, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [],         keyword  count earliest_timestamp example_message
0        socket      0                NaT                
1    connection      0                NaT                
2       timeout      0                NaT                
3     timed out      0                NaT                
4   broken pipe      0                NaT                
5       refused      0                NaT                
6          disk      0                NaT                
7            io      0                NaT                
8           i/o      0                NaT                
9         error      0                NaT                
10    exception      0                NaT                )```
```

In [ ]:
# In[5]:

import numpy as np

# Reuse variables from kernel: agg_summary_df, metrics_df, metrics_win, logs_df, errs_df, thresholds_df
# Candidate services
candidates = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# Ensure thresholds for metrics are available (global P95/P05 computed earlier)
metrics_thresholds = thresholds_df[thresholds_df['series_type'] == 'metric'].copy()
# Filter to candidate services
metrics_thresholds = metrics_thresholds[metrics_thresholds['cmdb_id'].isin(candidates)].rename(
    columns={'p95_full':'global_p95','p05_full':'global_p05'}
)[['series_type','cmdb_id','kpi_name','global_p95','global_p05']]

# Prepare windowed metrics (metrics_win already filtered to the incident window earlier)
# Ensure metrics_win exists; if not, create by filtering metrics_df
try:
    _ = metrics_win
except NameError:
    window_start = pd.to_datetime("2024-01-17 17:06:07", utc=True)
    window_end = pd.to_datetime("2024-01-17 17:36:07", utc=True)
    metrics_win = metrics_df[(metrics_df['timestamp'] >= window_start) & (metrics_df['timestamp'] <= window_end)].copy()
# Keep only candidate services in window data
metrics_win_cand = metrics_win[metrics_win['cmdb_id'].isin(candidates)].copy()

# Merge thresholds with window rows using right join semantics: keep all thresholded series (even if no window samples)
m = metrics_thresholds.merge(
    metrics_win_cand,
    on=['cmdb_id','kpi_name'],
    how='left',
    suffixes=('','_win')
)

# Flags for above/below using global thresholds
m['above95'] = m['value'].notna() & (m['value'] >= m['global_p95'])
m['below05'] = m['value'].notna() & (m['value'] <= m['global_p05'])
m['ts_above'] = m['timestamp'].where(m['above95'])
m['ts_below'] = m['timestamp'].where(m['below05'])

# Aggregate per series
group_cols = ['cmdb_id','kpi_name','global_p95','global_p05']
agg = m.groupby(group_cols).agg(
    points_in_window = ('value', 'count'),
    count_above_p95 = ('above95', 'sum'),
    earliest_above_p95 = ('ts_above', 'min'),
    max_value_in_window = ('value', 'max'),
    count_below_p05 = ('below05', 'sum'),
    earliest_below_p05 = ('ts_below', 'min'),
    min_value_in_window = ('value', 'min')
).reset_index()

# Ensure integer types for counts
for c in ['points_in_window','count_above_p95','count_below_p05']:
    agg[c] = agg[c].astype(int)

# Compute severity_ratio per rules
def compute_severity(row):
    gp95 = row['global_p95']
    mx = row['max_value_in_window']
    # handle NaN max -> treat as no window samples -> severity 1.0
    if pd.isna(mx):
        return 1.0
    if gp95 > 0:
        return float(mx) / float(gp95) if gp95 != 0 else 1.0
    # gp95 == 0
    if gp95 == 0 and mx > 0:
        return float('inf')
    return 1.0

agg['severity_ratio'] = agg.apply(compute_severity, axis=1)
agg['total_anom'] = agg['count_above_p95'] + agg['count_below_p05']

# Order and limit to top 20 by severity_ratio desc then total_anom desc
service_kpi_df_top20 = agg.sort_values(['severity_ratio','total_anom'], ascending=[False, False]).head(20).reset_index(drop=True)

# Select requested columns and keep compact
service_kpi_df_top20 = service_kpi_df_top20[
    ['cmdb_id','kpi_name','global_p95','global_p05','points_in_window',
     'count_above_p95','earliest_above_p95','max_value_in_window',
     'count_below_p05','earliest_below_p05','min_value_in_window','severity_ratio']
]

# --------------------------
# Part 2: logs and error_logs for candidate services in the window
window_start = pd.to_datetime("2024-01-17 17:06:07", utc=True)
window_end = pd.to_datetime("2024-01-17 17:36:07", utc=True)

# 2a) logs.csv aggregation per (cmdb_id, log_name)
logs_cand_win = logs_df[
    (logs_df['cmdb_id'].isin(candidates)) &
    (logs_df['timestamp'] >= window_start) &
    (logs_df['timestamp'] <= window_end)
].copy()

# ensure numeric
logs_cand_win['value'] = pd.to_numeric(logs_cand_win['value'], errors='coerce')

logs_agg = (
    logs_cand_win.groupby(['cmdb_id','log_name'])['value']
    .agg(points_in_window='count', mean_value_in_window='mean', min_value_in_window='min', max_value_in_window='max')
    .reset_index()
    .sort_values(['points_in_window','cmdb_id'], ascending=[False, True])
)

# 2b) error_logs rows up to 100 sorted ascending
errs_cand_win = errs_df[
    (errs_df['cmdb_id'].isin(candidates)) &
    (errs_df['timestamp'] >= window_start) &
    (errs_df['timestamp'] <= window_end)
].copy()

errs_cand_sorted = errs_cand_win.sort_values('timestamp', ascending=True).head(100).reset_index(drop=True)

# Keyword scan (case-insensitive substring, regex=False to handle 'i/o' literal)
keywords = ['socket','connection','timeout','timed out','broken pipe','refused','disk','io','i/o','error','exception']
kw_rows = []
for kw in keywords:
    mask = errs_cand_win['message'].str.contains(kw, case=False, na=False, regex=False)
    cnt = int(mask.sum())
    if cnt > 0:
        earliest = errs_cand_win.loc[mask, 'timestamp'].min()
        example = errs_cand_win.loc[mask].sort_values('timestamp').iloc[0]['message']
        example_trim = example[:200] if pd.notna(example) else ""
    else:
        earliest = pd.NaT
        example_trim = ""
    kw_rows.append({'keyword': kw, 'count': cnt, 'earliest_timestamp': earliest, 'example_message': example_trim})

errs_keyword_summary = pd.DataFrame(kw_rows).sort_values('count', ascending=False).reset_index(drop=True)

# Return the compact results
service_kpi_df_top20, logs_agg, errs_cand_sorted[['timestamp','cmdb_id','message']], errs_keyword_summary

```
Out[5]:
```
Summary — plain English

Window: 2024-01-17 17:06:07 → 2024-01-17 17:36:07 UTC for the five candidate services.

1) Per-service KPI highlights (most severe first)
- recommendationservice — latency-50: severity_ratio ≈ 2.36 (global P95 = 0.0269). 25 points in window; 2 samples >= P95 (earliest 2024-01-17 17:22:00), 2 samples <= P05 (earliest 17:28:00). Max in window ≈ 0.0633.
- productcatalogservice — latency-50: severity_ratio ≈ 1.55. 25 points; 2 above P95 (earliest 17:22:00), 2 below P05 (earliest 17:18:00). Max ≈ 0.00448.
- productcatalogservice — latency-90 and cpu also high in ranking (severity_ratio ~1.39 and ~1.35 respectively). Earliest high-latency samples around 17:22:00.
- checkoutservice — latency-50 and latency-90 have elevated severity_ratio (~1.34 and ~1.09). Earliest above-P95 samples around 17:21–17:22:00 for latency; some CPU/memory modestly elevated.
- currencyservice, emailservice — show smaller but measurable deviations (latency and workload/cpu/mem entries with severity_ratio ≈ 1.02–1.08). Earliest above-P95 often at ~17:17–17:22.

General pattern: most impacted KPIs are latency metrics (latency-50 and latency-90) across recommendationservice, productcatalogservice and checkoutservice, with earliest above-threshold samples often at ~2024-01-17 17:22:00 UTC. Severity ratios are modest (mostly 1.0–2.4), indicating values up to ~2.4× the global P95 in the window for the worst KPI.

2) Logs and error logs for the five services
- log.csv (in-window aggregates):
  - All services have log.error_count present but mean = 0.00 (25 points each) — no logged error counts during the window.
  - log.row_count present for each service with differing traffic: checkoutservice mean ~81.7, currencyservice mean ~2291.6, recommendationservice mean ~537.4, emailservice mean ~27.2.
- error_logs.csv:
  - No error_log rows returned for these services in the window (empty result).
  - Keyword scan over error messages found zero matches for all keywords (socket, connection, timeout, timed out, broken pipe, refused, disk, io, i/o, error, exception).

Interpretation and recommended next steps
- The telemetry shows latency spikes concentrated around ~17:22:00 affecting recommendationservice and productcatalogservice most strongly, with checkoutservice also showing latency and modest CPU/memory increases.
- There are no corresponding textual error logs or nonzero log.error_count for these services in the time window, so the issue appears in traces/metrics rather than being captured as explicit application error logs.
- Recommended actions:
  1. Drill into traces for recommendationservice/productcatalogservice/checkoutservice around 17:21–17:25 to find slow spans and dependency failures.
  2. Inspect upstream callers (frontend, root) and downstream dependencies for these services.
  3. Verify logging pipeline and log level to ensure errors (if any) are being captured; consider increasing trace/log detail if needed.
  4. Check resource and network metrics for nodes running productcatalogservice and recommendationservice (disk I/O, sockets, network latency) even though textual error matches are absent.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name    global_p95    global_p05  points_in_window  count_above_p95        earliest_above_p95  max_value_in_window  count_below_p05        earliest_below_p05  min_value_in_window  severity_ratio
0   recommendationservice  latency-50  2.688522e-02  7.054004e-03                25                2 2024-01-17 17:22:00+00:00         6.333290e-02                2 2024-01-17 17:28:00+00:00         6.986184e-03        2.355677
1   productcatalogservice  latency-50  2.890383e-03  1.657662e-03                25                2 2024-01-17 17:22:00+00:00         4.483068e-03                2 2024-01-17 17:18:00+00:00         1.652536e-03        1.551029
2   productcatalogservice  latency-90  6.230397e-02  4.343052e-03                25                2 2024-01-17 17:22:00+00:00         8.644342e-02                2 2024-01-17 17:31:00+00:00         4.338961e-03        1.387446
3   productcatalogservice         cpu  1.386276e+01  3.202403e+00                25                2 2024-01-17 17:22:00+00:00         1.875374e+01                2 2024-01-17 17:14:00+00:00         3.197929e+00        1.352814
4         checkoutservice  latency-50  1.399058e-01  6.452446e-02                25                2 2024-01-17 17:22:00+00:00         1.877048e-01                2 2024-01-17 17:18:00+00:00         6.218811e-02        1.341652
5   recommendationservice  latency-90  8.014818e-02  9.457782e-03                25                2 2024-01-17 17:22:00+00:00         9.288783e-02                2 2024-01-17 17:25:00+00:00         9.446067e-03        1.158951
6         checkoutservice  latency-90  3.381761e-01  1.867105e-01                25                2 2024-01-17 17:21:00+00:00         3.671317e-01                2 2024-01-17 17:13:00+00:00         1.730930e-01        1.085623
7         currencyservice  latency-50  3.680468e-03  3.439574e-03                25                2 2024-01-17 17:22:00+00:00         3.988391e-03                2 2024-01-17 17:19:00+00:00         3.401876e-03        1.083664
8            emailservice    workload  1.212893e+00  6.068900e-01                25                2 2024-01-17 17:24:00+00:00         1.305333e+00                2 2024-01-17 17:31:00+00:00         5.670000e-01        1.076214
9         checkoutservice    workload  1.218170e+00  6.213567e-01                25                2 2024-01-17 17:24:00+00:00         1.303200e+00                2 2024-01-17 17:31:00+00:00         5.670000e-01        1.069801
10        currencyservice  latency-90  3.260517e-02  1.187889e-02                25                2 2024-01-17 17:17:00+00:00         3.448768e-02                2 2024-01-17 17:22:00+00:00         9.479916e-03        1.057737
11        currencyservice    workload  8.332101e+01  7.597754e+01                25                2 2024-01-17 17:17:00+00:00         8.633440e+01                2 2024-01-17 17:13:00+00:00         7.513332e+01        1.036166
12           emailservice         cpu  3.261867e-01  2.722702e-01                25                2 2024-01-17 17:19:00+00:00         3.360128e-01                2 2024-01-17 17:13:00+00:00         2.564983e-01        1.030124
13  recommendationservice         cpu  2.714161e+00  2.385436e+00                25                2 2024-01-17 17:13:00+00:00         2.786036e+00                2 2024-01-17 17:17:00+00:00         2.341057e+00        1.026482
14        currencyservice         mem  4.619588e+07  4.531634e+07                25                2 2024-01-17 17:29:00+00:00         4.713933e+07                2 2024-01-17 17:18:00+00:00         4.524244e+07        1.020423
15        checkoutservice         cpu  5.075203e-01  3.253652e-01                25                2 2024-01-17 17:17:00+00:00         5.125432e-01                2 2024-01-17 17:20:00+00:00         2.829257e-01        1.009897
16           emailservice  latency-90  4.768933e-03  4.600000e-03                25                2 2024-01-17 17:16:00+00:00         4.813492e-03                9 2024-01-17 17:12:00+00:00         4.600000e-03        1.009344
17  productcatalogservice         mem  2.077934e+07  1.010674e+07                25                2 2024-01-17 17:24:00+00:00         2.095766e+07                2 2024-01-17 17:14:00+00:00         1.009814e+07        1.008582
18           emailservice  latency-50  3.093852e-03  3.000000e-03                25                2 2024-01-17 17:16:00+00:00         3.118607e-03                9 2024-01-17 17:12:00+00:00         3.000000e-03        1.008001
19        checkoutservice         mem  1.138235e+07  1.082191e+07                25                2 2024-01-17 17:24:00+00:00         1.147085e+07                2 2024-01-17 17:12:00+00:00         1.077378e+07        1.007775,                  cmdb_id         log_name  points_in_window  mean_value_in_window  min_value_in_window  max_value_in_window
0        checkoutservice  log.error_count                25                  0.00                    0                    0
1        checkoutservice    log.row_count                25                 81.72                   12                  114
2        currencyservice  log.error_count                25                  0.00                    0                    0
3        currencyservice    log.row_count                25               2291.56                  357                 2631
4           emailservice  log.error_count                25                  0.00                    0                    0
5           emailservice    log.row_count                25                 27.24                    4                   38
6  recommendationservice  log.error_count                25                  0.00                    0                    0
7  recommendationservice    log.row_count                25                537.40                   56                  594, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [],         keyword  count earliest_timestamp example_message
0        socket      0                NaT                
1    connection      0                NaT                
2       timeout      0                NaT                
3     timed out      0                NaT                
4   broken pipe      0                NaT                
5       refused      0                NaT                
6          disk      0                NaT                
7            io      0                NaT                
8           i/o      0                NaT                
9         error      0                NaT                
10    exception      0                NaT                )```
```